# result comparison of MCMC vs FM
This notebook is to compare the inferred posterior (conditioned on the same simulated lightcurve) resulting from MCMC sampling and a trained flow matching model 

In [1]:
import os
import sys

from corner import corner

sys.path.append('..')

from src.simulator import Model, BurstSimulator
from src.flow_matching.simulator import BatchedModel

from src.c2st import c2st
from src.flow_matching.loader import empty_classifier_from_config, empty_model_from_config, read_config, prob_path_from_config
from src.flow_matching.distributions import UniformPrior, CompositePrior, Posterior, DiscreteUniform
from src.flow_matching.probability_path import GuidedLinearProbabilityPath
from src.flow_matching.integration import EulerODESolver
from src.flow_matching.models import MLPGuidedVectorField, FRBLightCurveCNN, LightCurveThinner, fourier_embedding, LightCurveMLP, UNetEncoder, TransdimensionalModel, EncodedClassifier
from src.flow_matching.transformer import TransformerGuidedField
from src.helpers import record_every, plot_posterior_samples, gen_parameter_labels

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.lines as mlines

import numpy as np
import torch
import yaml

from src.flow_matching.plotting import plot_loss, plot_snapshots
from src.flow_matching.helpers import choose_device, build_mlp, find_run_dir

device = choose_device()

# Loading in the FM model

In [2]:
# fill in desired job_id or directory name 
job_id = "14044747" #False
run_dir = None
save_dir = "../checkpoints/"

In [3]:
# loading the model (via run_id, or path)
run_dir = find_run_dir(job_id, save_dir) if job_id else os.path.join(save_dir, run_dir)

checkpoint_path = os.path.join(run_dir, 'training_checkpoint.pth')
config_path     = os.path.join(run_dir, 'config.yaml')

In [ ]:
# create empty model from config
config = read_config(config_path)
vector_field = empty_model_from_config(config)

In [ ]:
transdimensional = not config['training']['fixed_N'] #False
print(transdimensional)
if transdimensional:
    classifier = empty_classifier_from_config(config)
    vector_field = TransdimensionalModel(classifier, vector_field)

In [ ]:
# load trained model 
print(checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Load 'normal' or EMA version 
ema = False
if ema:
    EMA_checkpoint_path = os.path.join(run_dir, "EMA_checkpoint.pth")
    state_dict = torch.load(EMA_checkpoint_path, weights_only=False)
    vector_field.load_state_dict(state_dict) 
else:
    vector_field.load_state_dict(checkpoint["model_state_dict"])

vector_field.eval()
vector_field.to(device)

losses = checkpoint["losses"]

In [ ]:
path = prob_path_from_config(config)
inf_params = config['model']['init_params']['inf_params']
N = path.p_data.model_params['ncomp']
vector_dim = N * len(inf_params)
burstparams = path.p_data.model_params['burstparams']

try:
    mean, std = torch.tensor(config['training']['sample_mean'], device=device), torch.tensor(config['training']['sample_std'], device=device)
except KeyError:
    mean, std = torch.zeros(vector_dim, device=device), torch.ones(vector_dim, device=device)

# Loading the matching MCMC samples

In [ ]:
# check if run with same N, inf_params and burstparams exists
def mcmc_settings_exist(settings, dictionary):
    print(settings)

    for key, value in dictionary.items():

        # key does not exits
        if not settings.get(key, False):
            return False
        if key == 'burstparams':
            same_params = compare_dicts(ignore_keys=settings['inf_params'], d1=settings[key], d2=value)
        elif key == 'inf_params':
            if not set(settings[key]) == set(value):
                return False
        elif key == 'N':
            if settings[key] != value:
                return False
        
    return True 

def compare_dicts(ignore_keys, d1, d2, rel_tol=1e-9, abs_tol=0.0):
    if d1.keys() != d2.keys():
        return False
    return np.all(np.isclose(d1[k], d2[k], rel_tol=rel_tol, abs_tol=abs_tol) for k in d1 if k not in ignore_keys)

def mcmc_run_exists(N, inf_params, burstparams, save_dir='../MCMC_runs'):
    dirs = os.listdir(save_dir)

    for dir in dirs:

        with open(os.path.join(save_dir, dir, 'settings.yaml'), 'r') as f:
            settings = yaml.safe_load(f)

        if mcmc_settings_exist(settings, {'N':N, 'inf_params':inf_params, 'burstparams':burstparams}):
            return os.path.join(save_dir, dir)
        
    return False

N_true = 3
run_path = mcmc_run_exists(N_true, inf_params, burstparams)
if not run_path:
    print("No matching MCMC run found")
else:
    print(f'matching mcmc run found at {run_path}!')
    settings_file = open(os.path.join(run_path, 'settings.yaml'), 'r')
    burstparams = yaml.safe_load(settings_file)['burstparams']
    settings_file.close()

    samples = np.load(os.path.join(run_path, 'samples.npy'))
    simulated_counts = np.load(os.path.join(run_path, 'simulated_counts.npy'))

In [ ]:
plt.plot(simulated_counts)

# generate FM posterior 

In [ ]:
num_samples = 25000  # number of prior samples to transform 
samples_per_batch = 5000
batches = num_samples // samples_per_batch
final_snapshot = torch.zeros((batches * samples_per_batch, vector_dim), device=device)
Ns = torch.zeros((batches * samples_per_batch, 1), device=device)
N = 7

# use same data point for conditioning all prior samples
condition = torch.tensor(simulated_counts / 150, device=device, dtype=torch.float)
simulations = condition.repeat(samples_per_batch, 1)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)

if transdimensional:
    logits = classifier(condition.unsqueeze(0))
    p_N = torch.softmax(logits, dim=1).flatten()
else:
    p_N = torch.zeros(N, device=device)
    p_N[N-1] = 1
    
# integrate in batches
for i in range(batches):
    # simulate ODE starting from x0
    x0 = (path.p_simple.sample(samples_per_batch).to(device) - mean) / std

    start = i * samples_per_batch
    stop = start + samples_per_batch

    N_samples = torch.multinomial(p_N.repeat(samples_per_batch, 1), num_samples=1) + 1
    Ns[start:stop, :] = N_samples
    final_snapshot[start:stop, :] = solver.solve(x0, ts.view(1, nts, 1).expand(samples_per_batch, nts, 1), y=simulations, N=N_samples) * std + mean

In [ ]:
final_snapshot

In [ ]:
# NOTE : IF TRANSDIMENSIONAL, HAVE TO CUT OFF IRRELEVANT TOKENS FIRST, THEN SORT.
sorted_snapshot = final_snapshot.reshape(-1, len(inf_params), N)

# sort param columns based on peaktime row
peaktime_idx = inf_params.index('t0')

indeces = sorted_snapshot[:, peaktime_idx, :].argsort(dim=-1)
indeces_expanded = indeces.unsqueeze(1).expand(-1, len(inf_params), -1)

sorted_snapshot = torch.gather(sorted_snapshot, dim=-1, index=indeces_expanded)
sorted_snapshot = sorted_snapshot.view(-1, vector_dim)
sorted_snapshot

In [ ]:
# plot sorted peaktimes
plt.figure(figsize=(20, 3))
indeces = np.random.choice(range(25000), 200)
plt.plot(sorted_snapshot[indeces][:, :1].detach().cpu().numpy(), 'o')
plt.plot(sorted_snapshot[indeces][:, 1:2].detach().cpu().numpy(), 'o')
plt.plot(sorted_snapshot[indeces][:, 2:3].detach().cpu().numpy(), 'o')

In [ ]:
p_N

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

# Corner plot overlay

In [ ]:
range_ = [1, 1]
samples=None
N_inf = 3  # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf
# FM_samples = sorted_snapshot[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf
FM_samples = final_snapshot[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf

# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
N=7
print(FM_samples[1], N)
new_dim = N_inf * len(inf_params)
FM_samples = FM_samples.view(-1, len(inf_params), N)[:, :, :N_inf].reshape(bs, new_dim)  
print(FM_samples[1], FM_samples.shape)
data = [FM_samples.cpu().numpy(), samples]
colors=['blue', 'red']
labels=['FM', 'MCMC']

var_names = gen_parameter_labels(inf_params, N)
var_names = np.array(var_names).reshape(len(inf_params), N)[:, :N_inf].flatten()
true_values = np.array([burstparams[key][:N_inf] for key in inf_params]).flatten()

fig = None
for i in range(1):#len(data)):
    fig = corner(data[i], labels=var_names, truths=true_values, range=[range_[i] for _ in range(N_inf * len(inf_params))], 
             color='black', label=labels[i], fig=fig, plot_density=True, plot_datapoints=True, 
             fill_contours=False, plot_contours=True, hist_kwargs={'density':True}, bins=20)

# make legend with dummy lines
plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=labels[i])
            for i in range(len(data))
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, 2), loc="upper right"
    )
plt.show()


# Posterior samples

In [ ]:
# posterior samples conditioned on N=N_inf
burstparams_inf = {key:value[:N_inf] for key, value in burstparams.items()}
print(burstparams_inf, N_inf)
modelparams = {'time':torch.linspace(0, 1, 1000), 'burstparams':burstparams_inf, 'ybkg':5,'ncomp':3}
true_flux = Model(**modelparams).get_flux()
# true_flux=None
plt.figure(figsize=(12,5))

burstparams_inf = {'t0': [0.3, 0.5, 0.7, 0.6], 'amp': [2.0, 2.0, 2.0, 0.0], 'rise': [-1.52, -1.52, -1.5228787452803376, -1.5228787452803376], 'skew': [5.0, 5.0, 5.0, 5.0]}
# print(burstparams_inf)

for i in range(1):

    plt.subplot(121 + i)

    plot_posterior_samples(
        1000, 
        simulated_counts, 
        data[i], 
        inf_params, 
        modelparams={'time':np.linspace(0, 1, 1000), 'burstparams':burstparams_inf, 'ybkg':5,'ncomp':N_inf},
        true_flux=true_flux,
        title=f"100 {labels[i]} posterior samples"
        )

In [ ]:
# posterior samples that include all sampled component numbers
modelparams = {'time':torch.linspace(0, 1, 1000), 'burstparams':burstparams, 'ybkg':5,'ncomp':3}
true_flux = Model(**modelparams).get_flux()
plt.plot(simulated_counts, alpha=0.3)
plt.plot(true_flux)
posterior_param_samples = path.p_simple.samples_as_dict(final_snapshot)
posterior_curve_samples = BatchedModel(device=device, **{'time':torch.linspace(0, 1, 1000), 'burstparams':posterior_param_samples, 'ybkg':5,'ncomp':Ns}).get_flux()
for i in range(5500, 6000):
    plt.plot(posterior_curve_samples[i].cpu(), alpha=0.2)

# MSE and CEL loss

In [ ]:
if transdimensional:
    MSE_loss = checkpoint['MSE_loss']
    CEL_loss = checkpoint['CEL_loss']
    plt.loglog(MSE_loss, label='MSE')
    plt.loglog(CEL_loss, label='CEL')
    plt.legend()

# p(N) for collection of lightcurves

In [ ]:
examples = 15
params, counts, components = path.p_data.sample(examples)

# path.p_simple.priors['rise'].x_max = -1
burstparams = path.p_simple.samples_as_dict(params)

model = BatchedModel(
            time=torch.linspace(0, 1, 1000), 
            ncomp=components, 
            ybkg=5.0, 
            burstparams=burstparams,
            device=device
            )
true_flux = model.get_flux()
condition = torch.tensor(counts / 150, device=device, dtype=torch.float)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)

logits = classifier(condition)
p_N = torch.softmax(logits, dim=1)

for i in range(examples):
    plt.figure(figsize=(15, 5))
    plt.subplot(121)
    plt.bar(range(1, 8), height=p_N[i].flatten().cpu().detach().numpy())
    plt.xlabel("$N_{pred}$")
    plt.title('p(N|y)')

    print(path.p_simple.samples_as_dict(params[i].view(1,-1)))

    plt.subplot(122)
    plt.title(f'N_true = {components[i].item()}')
    plt.plot(counts[i], 'k-')
    plt.plot(true_flux[i].cpu(), 'm-', alpha=0.7)


# C2ST

Performs classifier two-sample test. A binary classifier is trained to distinguish samples from MCMC and FM, and its accuracy is evaluated. If the accuracy is 50\%, this indicates the classifier is unable to distinguish between the two distributions. If the accuracy is 100\%, the classifier can easily distinguish between samples from MCMC and FM. I.e. lower score is better and the 'best' minimum score is 50%. 

In [ ]:
# ensure number of MCMC samples equal to FM
MCMC_samples = np.copy(samples)
np.random.shuffle(MCMC_samples)
sample_size, _ = FM_samples.shape
MCMC_samples = torch.tensor(MCMC_samples[:sample_size], device=device, dtype=torch.float32)

# standardize using common mean and std
concat = torch.cat((FM_samples, MCMC_samples))
mean  = torch.mean(concat)
std  = torch.std(concat)

MCMC_samples = (MCMC_samples - mean) / std 
FM_samples   = (FM_samples - mean) / std 

accuracy = c2st(FM_samples, MCMC_samples)

In [ ]:
print(f'c2st score: {accuracy}')